In [1]:
import pandas as pd
import numpy as np

forecast_results = pd.read_csv(
    "../data/processed/forecast_results.csv",
    parse_dates=["date"]
)

forecast_results.head()

,date,item,actual_demand,q10,q50,q90
0,2017-07-09,1,2227,2221.549515,2277.337242,2391.773561
1,2017-07-16,1,2261,2107.127532,2248.020940,2320.767430
2,2017-07-23,1,2239,2113.073516,2284.675439,2361.518382
3,2017-07-30,1,2247,2204.192185,2281.343412,2349.492661
4,2017-08-06,1,1947,1921.010004,2112.598400,2178.813692


In [2]:
forecast_results["uncertainty"] = (
    forecast_results["q90"]
    - forecast_results["q50"]
)

forecast_results[
    ["date", "item", "q10", "q50", "q90", "uncertainty"]
].head(10)

,date,item,q10,q50,q90,uncertainty
0,2017-07-09,1,2221.549515,2277.337242,2391.773561,114.436320
1,2017-07-16,1,2107.127532,2248.020940,2320.767430,72.746490
2,2017-07-23,1,2113.073516,2284.675439,2361.518382,76.842943
3,2017-07-30,1,2204.192185,2281.343412,2349.492661,68.149249
4,2017-08-06,1,1921.010004,2112.598400,2178.813692,66.215292
5,2017-08-13,1,1899.368354,1926.751374,2014.034092,87.282718
6,2017-08-20,1,1880.364806,1914.420823,1975.721328,61.300505
7,2017-08-27,1,1871.086582,1971.616185,2020.716490,49.100304
8,2017-09-03,1,1895.740242,1899.924116,1937.788858,37.864741
9,2017-09-10,1,1784.360910,1818.433110,1909.377057,90.943946


In [3]:
forecast_results["safety_stock"] = (
    0.5 * forecast_results["uncertainty"]
)

In [4]:
forecast_results["planning_demand"] = (
    forecast_results["q50"]
    + forecast_results["safety_stock"]
)

In [5]:
forecast_results[
    [
        "q50",
        "q90",
        "uncertainty",
        "safety_stock",
        "planning_demand"
    ]
].describe()

,q50,q90,uncertainty,safety_stock,planning_demand
count,520.000000,520.000000,520.000000,520.000000,520.000000
mean,4452.432009,4581.699739,129.267730,64.633865,4517.065874
std,1933.945160,1970.457330,99.202700,49.601350,1951.656397
min,1055.957812,1112.327055,-218.331161,-109.165580,1084.142433
25%,2774.726452,2901.414318,67.116745,33.558373,2852.567261
50%,4522.109222,4676.844299,112.899252,56.449626,4603.771054
75%,5931.776292,6120.547752,183.887300,91.943650,6028.543270
max,8284.315804,8621.649056,654.803206,327.401603,8443.723027


In [6]:
print(
    "Average q50:",
    forecast_results["q50"].mean()
)

print(
    "Average safety stock:",
    forecast_results["safety_stock"].mean()
)

print(
    "Average planning demand:",
    forecast_results["planning_demand"].mean()
)

Average q50: 4452.432008507957
Average safety stock: 64.63386522674726
Average planning demand: 4517.065873734705


In [7]:
items = sorted(forecast_results["item"].unique())
weeks = sorted(forecast_results["date"].unique())

planning_demand = (
    forecast_results
    .pivot(index="date", columns="item", values="planning_demand")
    .loc[weeks, items]
)

planning_demand.head()

item,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20
date,,,,,,,,,,,,,,,,,,,,
2017-07-09,2334.555401,5933.857433,3730.861262,2266.448425,1886.437776,6068.607282,5889.940002,7857.069808,5298.011343,7496.468591,7122.207237,7045.208182,8391.101646,5871.245861,8425.932916,2644.850016,3325.129132,8436.994847,4112.841666,4853.963718
2017-07-16,2284.394185,5941.145860,3813.347941,2277.742043,1903.406411,6045.322137,6045.988796,7928.935974,5275.756683,7520.308832,7224.896620,7246.195444,8374.774056,6009.773168,8402.281599,2663.529478,3368.885255,8390.213887,4189.071957,4918.238812
2017-07-23,2323.096911,5991.916970,3871.912195,2300.776581,1880.167261,6089.984820,6012.145955,7916.043925,5309.337443,7471.977381,7161.922023,7197.385029,8374.864465,6042.016997,8435.396410,2658.876942,3401.036253,8401.173996,4044.640162,4907.876113
2017-07-30,2315.418037,6017.320711,3802.556723,2236.546735,1929.912047,6142.637891,6146.656374,8024.022134,5263.546546,7543.978163,7324.204145,7403.807103,8381.914531,6121.758247,8436.727193,2705.797059,3438.496631,8443.723027,4050.530265,4894.803681
2017-08-06,2145.706046,5749.285082,3543.427021,2154.810962,1794.698064,5734.913836,5762.747291,7398.485820,4950.384191,6802.312619,6676.283335,6595.606859,8234.485549,5696.647788,8336.010677,2503.164933,3235.209095,8095.982116,3838.506403,4607.746365


In [11]:
import pulp

In [12]:
import pulp
print(pulp.__version__)

4.0.0


In [13]:
model_uncertain = pulp.LpProblem(
    "Uncertainty_Aware_Planner",
    pulp.LpMinimize
)

In [19]:
import pandas as pd
import numpy as np
import pulp

forecast_results = pd.read_csv(
    "../data/processed/forecast_results.csv",
    parse_dates=["date"]
)

items = sorted(forecast_results["item"].unique())
weeks = sorted(forecast_results["date"].unique())

print(len(items), "items")
print(len(weeks), "weeks")

20 items
26 weeks


In [20]:
forecast_results["uncertainty"] = (
    forecast_results["q90"] - forecast_results["q50"]
)

forecast_results["safety_stock"] = (
    0.5 * forecast_results["uncertainty"]
)

forecast_results["planning_demand"] = (
    forecast_results["q50"]
    + forecast_results["safety_stock"]
)

planning_demand = (
    forecast_results
    .pivot(index="date", columns="item", values="planning_demand")
    .loc[weeks, items]
)

print("Planning demand ready.")

Planning demand ready.


In [21]:
model_uncertain = pulp.LpProblem(
    "Uncertainty_Aware_Planner",
    pulp.LpMinimize
)

In [22]:
production_u = {}
inventory_u = {}
backlog_u = {}

for item in items:
    for week in weeks:

        production_u[(item, week)] = model_uncertain.add_variable(
            f"Production_{item}_{week}",
            lowBound=0,
            cat="Continuous"
        )

        inventory_u[(item, week)] = model_uncertain.add_variable(
            f"Inventory_{item}_{week}",
            lowBound=0,
            cat="Continuous"
        )

        backlog_u[(item, week)] = model_uncertain.add_variable(
            f"Backlog_{item}_{week}",
            lowBound=0,
            cat="Continuous"
        )

print("Variables created successfully.")


Variables created successfully.


In [23]:
for item in items:
    for i, week in enumerate(weeks):

        if i == 0:
            previous_inventory = 0
            previous_backlog = 0
        else:
            previous_week = weeks[i - 1]
            previous_inventory = inventory_u[(item, previous_week)]
            previous_backlog = backlog_u[(item, previous_week)]

        model_uncertain += (
            inventory_u[(item, week)]
            - backlog_u[(item, week)]
            ==
            previous_inventory
            - previous_backlog
            + production_u[(item, week)]
            - planning_demand.loc[week, item]
        )

print("Inventory balance constraints added.")

Inventory balance constraints added.


In [25]:
production_capacity = 90000

raw_material_capacity = {
    "RM1": 50000,
    "RM2": 75000,
    "RM3": 30000
}

production_cost = {
    1: 10, 2: 11, 3: 9, 4: 12, 5: 10,
    6: 11, 7: 9, 8: 12, 9: 10, 10: 11,
    11: 9, 12: 12, 13: 10, 14: 11, 15: 9,
    16: 12, 17: 10, 18: 11, 19: 9, 20: 12
}

holding_cost = 2
backlog_penalty = 25

bom = {
    1:  {"RM1": 0.5, "RM2": 0.8, "RM3": 0.2},
    2:  {"RM1": 0.6, "RM2": 0.7, "RM3": 0.3},
    3:  {"RM1": 0.4, "RM2": 0.9, "RM3": 0.2},
    4:  {"RM1": 0.7, "RM2": 0.6, "RM3": 0.4},
    5:  {"RM1": 0.5, "RM2": 0.8, "RM3": 0.3},
    6:  {"RM1": 0.6, "RM2": 0.7, "RM3": 0.2},
    7:  {"RM1": 0.4, "RM2": 0.8, "RM3": 0.3},
    8:  {"RM1": 0.7, "RM2": 0.9, "RM3": 0.2},
    9:  {"RM1": 0.5, "RM2": 0.6, "RM3": 0.4},
    10: {"RM1": 0.6, "RM2": 0.8, "RM3": 0.3},
    11: {"RM1": 0.4, "RM2": 0.7, "RM3": 0.2},
    12: {"RM1": 0.7, "RM2": 0.8, "RM3": 0.3},
    13: {"RM1": 0.5, "RM2": 0.9, "RM3": 0.2},
    14: {"RM1": 0.6, "RM2": 0.6, "RM3": 0.4},
    15: {"RM1": 0.4, "RM2": 0.8, "RM3": 0.3},
    16: {"RM1": 0.7, "RM2": 0.7, "RM3": 0.2},
    17: {"RM1": 0.5, "RM2": 0.8, "RM3": 0.4},
    18: {"RM1": 0.6, "RM2": 0.9, "RM3": 0.3},
    19: {"RM1": 0.4, "RM2": 0.6, "RM3": 0.2},
    20: {"RM1": 0.7, "RM2": 0.8, "RM3": 0.3}
}

print("Optimizer assumptions loaded.")

Optimizer assumptions loaded.


In [26]:
for week in weeks:
    model_uncertain += (
        pulp.lpSum(
            production_u[(item, week)]
            for item in items
        )
        <= production_capacity
    )

print("Capacity constraints added.")

Capacity constraints added.


In [27]:
for week in weeks:
    for material in ["RM1", "RM2", "RM3"]:
        model_uncertain += (
            pulp.lpSum(
                production_u[(item, week)] * bom[item][material]
                for item in items
            )
            <= raw_material_capacity[material]
        )

print("Raw-material constraints added.")

Raw-material constraints added.


In [28]:
total_cost_u = (
    pulp.lpSum(
        production_u[(item, week)] * production_cost[item]
        for item in items
        for week in weeks
    )
    +
    pulp.lpSum(
        inventory_u[(item, week)] * holding_cost
        for item in items
        for week in weeks
    )
    +
    pulp.lpSum(
        backlog_u[(item, week)] * backlog_penalty
        for item in items
        for week in weeks
    )
)

model_uncertain += total_cost_u

print("Objective function added.")

Objective function added.


In [29]:
solver = pulp.HiGHS()

result_u = model_uncertain.solve(solver)

print(result_u)

solver: HiGHS
status: Optimal (1)
has solution: True
objective: 86310893.895579
time: 0.0403s (cpu 0.0781s)


In [30]:
optimized_uncertain = []

for week in weeks:
    for item in items:
        optimized_uncertain.append({
            "date": week,
            "item": item,
            "planning_demand": planning_demand.loc[week, item],
            "production": production_u[(item, week)].value(),
            "inventory": inventory_u[(item, week)].value(),
            "backlog": backlog_u[(item, week)].value()
        })

optimized_uncertain = pd.DataFrame(optimized_uncertain)

optimized_uncertain.head()

,date,item,planning_demand,production,inventory,backlog
0,2017-07-09,1,2334.555401,2334.555401,0.0,0.0
1,2017-07-09,2,5933.857433,5933.857433,0.0,0.0
2,2017-07-09,3,3730.861262,3730.861262,0.0,0.0
3,2017-07-09,4,2266.448425,2266.448425,0.0,0.0
4,2017-07-09,5,1886.437776,1886.437776,0.0,0.0


In [31]:
optimized_uncertain.to_csv(
    "../data/processed/optimized_uncertain_plan.csv",
    index=False
)

print("Uncertainty-aware plan saved!")

Uncertainty-aware plan saved!


In [33]:
day5_sim = pd.read_csv(
    "../data/processed/simulation_results.csv",
    parse_dates=["date"]
)

day5_sim.head()

,date,item,forecast_demand,actual_demand,production,sales,ending_inventory,stockout
0,2017-07-09,1,2277.337242,2390.455805,2277.337242,2277.337242,0.0,113.118564
1,2017-07-16,1,2248.020940,2577.501852,2248.020940,2248.020940,0.0,329.480912
2,2017-07-23,1,2284.675439,2453.391085,0.000000,0.000000,0.0,2453.391085
3,2017-07-30,1,2281.343412,2172.027313,0.000000,0.000000,0.0,2172.027313
4,2017-08-06,1,2112.598400,2066.190552,0.000000,0.000000,0.0,2066.190552


In [34]:
uncertain_simulation = []

for item in sorted(optimized_uncertain["item"].unique()):

    item_plan = (
        optimized_uncertain[
            optimized_uncertain["item"] == item
        ]
        .sort_values("date")
    )

    # Same actual demand scenario used on Day 5
    item_actual = (
        day5_sim[
            day5_sim["item"] == item
        ]
        .sort_values("date")
    )

    inventory = 0

    for (_, plan_row), (_, actual_row) in zip(
        item_plan.iterrows(),
        item_actual.iterrows()
    ):

        available = inventory + plan_row["production"]
        actual_demand = actual_row["actual_demand"]

        sales = min(available, actual_demand)

        ending_inventory = available - sales
        stockout = actual_demand - sales

        uncertain_simulation.append({
            "date": plan_row["date"],
            "item": item,
            "planning_demand": plan_row["planning_demand"],
            "actual_demand": actual_demand,
            "production": plan_row["production"],
            "sales": sales,
            "ending_inventory": ending_inventory,
            "stockout": stockout
        })

        inventory = ending_inventory

uncertain_simulation = pd.DataFrame(
    uncertain_simulation
)

uncertain_simulation.head()

,date,item,planning_demand,actual_demand,production,sales,ending_inventory,stockout
0,2017-07-09,1,2334.555401,2390.455805,2334.555401,2334.555401,0.000000,55.900404
1,2017-07-16,1,2284.394185,2577.501852,0.000000,0.000000,0.000000,2577.501852
2,2017-07-23,1,2323.096911,2453.391085,4607.491096,2453.391085,2154.100011,0.000000
3,2017-07-30,1,2315.418037,2172.027313,2315.418037,2172.027313,2297.490734,0.000000
4,2017-08-06,1,2145.706046,2066.190552,2145.706046,2066.190552,2377.006228,0.000000


In [35]:
uncertain_service_level = (
    uncertain_simulation["sales"].sum()
    / uncertain_simulation["actual_demand"].sum()
)

uncertain_stockout_rate = (
    uncertain_simulation["stockout"].sum()
    / uncertain_simulation["actual_demand"].sum()
)

print(f"Uncertainty-Aware Service Level: {uncertain_service_level:.2%}")
print(f"Uncertainty-Aware Stockout Rate: {uncertain_stockout_rate:.2%}")

Uncertainty-Aware Service Level: 84.30%
Uncertainty-Aware Stockout Rate: 15.70%


In [36]:
uncertain_production_cost = sum(
    row["production"] * production_cost[row["item"]]
    for _, row in uncertain_simulation.iterrows()
)

uncertain_inventory_cost = (
    uncertain_simulation["ending_inventory"].sum()
    * holding_cost
)

uncertain_stockout_cost = (
    uncertain_simulation["stockout"].sum()
    * backlog_penalty
)

uncertain_total_cost = (
    uncertain_production_cost
    + uncertain_inventory_cost
    + uncertain_stockout_cost
)

print(f"Production Cost: {uncertain_production_cost:,.2f}")
print(f"Inventory Cost: {uncertain_inventory_cost:,.2f}")
print(f"Stockout Cost: {uncertain_stockout_cost:,.2f}")
print(f"Total Cost: {uncertain_total_cost:,.2f}")

Production Cost: 24,525,361.35
Inventory Cost: 9,943,529.97
Stockout Cost: 9,097,954.16
Total Cost: 43,566,845.47


In [37]:
uncertain_simulation.to_csv(
    "../data/processed/uncertainty_simulation_results.csv",
    index=False
)

print("Day 6 uncertainty simulation saved!")

Day 6 uncertainty simulation saved!
